# SafeLink — Complete Training Pipeline
**PRD v5.1 | 22-feature set | CNN+Dense + Autoencoder**

---

## ⚙️ Before You Start — One-time Setup

Your setup: **GTX 1660 Ti + CUDA 12.1 (already installed)**

You still need to install **cuDNN** and the correct **TensorFlow** version.

### Step 1 — Install cuDNN for CUDA 12.x
1. Go to: https://developer.nvidia.com/rdp/cudnn-download (NVIDIA account required)
2. Download: **cuDNN v8.9.x for CUDA 12.x** → Windows installer (`.exe`) or ZIP
3. If ZIP: copy the 3 folders (`bin/`, `include/`, `lib/`) into `C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\`
4. If `.exe`: run the installer, it does this automatically

### Step 2 — Install Python packages
Open a terminal (CMD or PowerShell) and run:
```
pip install tensorflow==2.15.0 pandas scikit-learn tqdm matplotlib
```
TF 2.15 is the recommended version for CUDA 12.x on Windows.

### Step 3 — Verify CUDA is on PATH
Run Section 1 below. If the GPU shows as `Not found`, run Section 1 Cell 3 to set the paths manually.

---

**Expected training time on GTX 1660 Ti (6 GB VRAM):**
| Step | Time |
|---|---|
| Section 2 — Data collection | 20–40 min |
| Section 3 — CNN training | 30–60 min |
| Section 4 — Autoencoder | 5–15 min |
| Section 5 — Copy to Android | < 1 min |

---
## Section 1 — Environment Check

In [ ]:
# Cell 1.1 — Check Python and TensorFlow versions
import sys
print(f'Python : {sys.version}')

try:
    import tensorflow as tf
    print(f'TensorFlow : {tf.__version__}')
    if tf.__version__ < '2.15':
        print('⚠️  TF < 2.15 detected. Run:  pip install tensorflow==2.15.0')
    else:
        print('✅ TensorFlow version is compatible with CUDA 12.1')
except ImportError:
    print('❌ TensorFlow not installed. Run:  pip install tensorflow==2.15.0')

In [ ]:
# Cell 1.2 — Detect GPU and enable memory growth
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)   # prevent OOM on 6 GB VRAM
        print(f'✅ GPU found : {g.name}')
    # Print CUDA / cuDNN build info
    print(f'   Built with CUDA : {tf.test.is_built_with_cuda()}')
    print(f'   CUDA version    : {tf.sysconfig.get_build_info().get("cuda_version", "unknown")}')
    print(f'   cuDNN version   : {tf.sysconfig.get_build_info().get("cudnn_version", "unknown")}')
else:
    print('❌ No GPU detected by TensorFlow.')
    print()
    print('   Common causes on Windows + CUDA 12.1:')
    print('   1. cuDNN not installed — see Step 1 in the header above')
    print('   2. CUDA/cuDNN not on PATH — run Cell 1.3 below to fix')
    print('   3. TF version mismatch — make sure tensorflow==2.15.0 is installed')
    print()
    print('   Quick check in a new terminal:')
    print('     nvidia-smi                     <- should show GTX 1660 Ti')
    print('     python -c "import tensorflow as tf; print(tf.config.list_physical_devices()"')

In [ ]:
# Cell 1.3 — ONLY run this if GPU was NOT detected in Cell 1.2
# Manually adds CUDA 12.1 to PATH so TF can find the libraries
import os

CUDA_PATH  = r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1'
CUDNN_PATH = r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin'

for p in [CUDA_PATH, CUDNN_PATH, os.path.join(CUDA_PATH, 'bin'), os.path.join(CUDA_PATH, 'libnvvp')]:
    if p not in os.environ.get('PATH', '') and os.path.isdir(p):
        os.environ['PATH'] = p + os.pathsep + os.environ.get('PATH', '')
        print(f'Added to PATH: {p}')

# Reload TF to pick up new PATH
import importlib, tensorflow
importlib.reload(tensorflow)
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(f'✅ GPU now detected: {[g.name for g in gpus]}')
else:
    print('❌ Still no GPU. Verify cuDNN files exist in:', CUDNN_PATH)
    print('   Check that cudnn64_8.dll (or similar) is in that folder.')

In [ ]:
# Cell 1.4 — Working directory and package check
import os, importlib

# Ensure we run from ml_pipeline/
cwd = os.path.abspath('')
if not cwd.endswith('ml_pipeline'):
    candidate = os.path.join(cwd, 'ml_pipeline')
    if os.path.isdir(candidate):
        os.chdir(candidate)
print(f'Working directory : {os.getcwd()}')
assert os.path.exists('feature_extractor.py'), '❌ feature_extractor.py not found'

# Package check
for pkg in ['pandas', 'sklearn', 'tqdm', 'matplotlib']:
    ok = importlib.util.find_spec(pkg) is not None
    print(f'  {"✅" if ok else "❌"} {pkg}')

print('\n✅ Environment ready')

---
## Section 2 — Data Collection

Loads raw CSVs from `../Datasets/`, extracts 22 features per URL, saves `data/safelink_dataset.csv`.

**Skip this section if `data/safelink_dataset.csv` already exists.**

| File (in `../Datasets/`) | Rows | Label |
|---|---|---|
| `malicious_phish_2021.csv` | ~651K | benign→0, phishing/malware→2, defacement→DROP |
| `PhiUSIIL_Phishing_URL_Dataset.csv` | ~235K | 1→0, 0→2 |
| `phishtank.csv` | ~7K | 2 |
| `openphish.txt` | varies | 2 |
| `tranco_top1m.csv` | top 10K | 0 |
| `urlhaus.csv` | ~170K | 2 |

In [ ]:
# Cell 2.1 — Check which dataset files are present
import os

DATASETS_DIR = '../Datasets'
expected = [
    'malicious_phish_2021.csv',
    'PhiUSIIL_Phishing_URL_Dataset.csv',
    'phishtank.csv',
    'openphish.txt',
    'tranco_top1m.csv',
    'urlhaus.csv',
]
print(f'Dataset folder: {os.path.abspath(DATASETS_DIR)}\n')
for fname in expected:
    path = os.path.join(DATASETS_DIR, fname)
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1024 / 1024
        print(f'  ✅  {fname:48s}  ({mb:.1f} MB)')
    else:
        print(f'  ⚠️   {fname:48s}  NOT FOUND — will be skipped')

In [ ]:
# Cell 2.2 — Run data collector
# Skips automatically if output file already exists.
import time, runpy, pandas as pd

OUTPUT_CSV = 'data/safelink_dataset.csv'
os.makedirs('data', exist_ok=True)

if os.path.exists(OUTPUT_CSV):
    n = len(pd.read_csv(OUTPUT_CSV, usecols=['label']))
    print(f'ℹ️  {OUTPUT_CSV} already exists ({n:,} rows) — skipping.')
    print('   Delete the file and re-run to force re-collection.')
else:
    t0 = time.time()
    runpy.run_path('data_collector.py', run_name='__main__')
    print(f'\n⏱  Done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 2.3 — Dataset sanity check
import pandas as pd
from feature_extractor import FEATURE_COLUMNS, N_FEATURES

df_check = pd.read_csv(OUTPUT_CSV, low_memory=False)
dist = df_check['label'].value_counts()

print(f'Total rows     : {len(df_check):,}')
print(f'Columns        : {df_check.shape[1]}  (expected 24 = url + label + 22 features)')
print(f'SAFE (0)       : {dist.get(0, 0):>10,}')
print(f'MALICIOUS (2)  : {dist.get(2, 0):>10,}')

assert df_check.shape[1] == 24, f'Expected 24 columns, got {df_check.shape[1]}'
assert N_FEATURES == 22, f'N_FEATURES must be 22, got {N_FEATURES}'
print('\n✅ Dataset OK')

---
## Section 3 — CNN + Dense Model Training

Trains the dual-stream model and exports `models/safelink_model.tflite` + `models/scaler_params.json`.

**Architecture (PRD v5.1):**
```
Stream A (Character CNN):  Embedding(100,32) → Conv1D(128,k=2) → Conv1D(128,k=3) → Conv1D(64,k=5) → GlobalMaxPool → Dense(128) → Dropout(0.3)
Stream B (Dense numeric):  BatchNorm → Dense(128) → Dense(64) → Dropout(0.2)
Fusion:                    Dense(256) → Dropout(0.3) → Dense(128) → Dropout(0.2) → Dense(64) → Softmax(3)
```

In [ ]:
# Cell 3.1 — Imports and constants
import os, json, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from urllib.parse import urlparse
import re
from feature_extractor import FEATURE_COLUMNS, N_FEATURES

# ── Paths ──────────────────────────────────────────────────────────
DATA_PATH   = 'data/safelink_dataset.csv'
OUTPUT_DIR  = 'models'
TFLITE_PATH = os.path.join(OUTPUT_DIR, 'safelink_model.tflite')
SCALER_PATH = os.path.join(OUTPUT_DIR, 'scaler_params.json')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Hyperparameters ─────────────────────────────────────────────────
SEQ_LEN    = 200
VOCAB_SIZE = 100
EMBED_DIM  = 32
BATCH_SIZE = 256    # fits 6 GB VRAM on GTX 1660 Ti with headroom
EPOCHS     = 30
LR         = 1e-3
SEED       = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'N_FEATURES = {N_FEATURES}  BATCH_SIZE = {BATCH_SIZE}')
assert N_FEATURES == 22

# Confirm GPU is active for this session
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU : {gpus[0].name if gpus else "NOT FOUND — training will use CPU"}')

In [ ]:
# Cell 3.2 — URL tokenizer (mirrors UrlTokenizer.kt)
def encode_url(url: str, seq_len: int = SEQ_LEN) -> list:
    """char -> (ord - 31).coerceIn(1, 99), zero-pad to seq_len."""
    encoded = [max(1, min(99, ord(c) - 31)) for c in url[:seq_len]]
    encoded += [0] * (seq_len - len(encoded))
    return encoded

print('encode_url ready')

In [ ]:
# Cell 3.3 — Model definition (PRD v5.1)
def build_model() -> keras.Model:
    # Stream A — Character CNN
    seq_input = keras.Input(shape=(SEQ_LEN,), dtype='int32', name='seq_input')
    x = keras.layers.Embedding(VOCAB_SIZE + 1, EMBED_DIM, mask_zero=True)(seq_input)
    x = keras.layers.Conv1D(128, kernel_size=2, activation='relu', padding='same')(x)
    x = keras.layers.Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = keras.layers.Conv1D(64,  kernel_size=5, activation='relu', padding='same')(x)
    x = keras.layers.GlobalMaxPooling1D()(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)

    # Stream B — Dense numeric
    num_input = keras.Input(shape=(N_FEATURES,), dtype='float32', name='num_input')
    y = keras.layers.BatchNormalization()(num_input)
    y = keras.layers.Dense(128, activation='relu')(y)
    y = keras.layers.Dense(64,  activation='relu')(y)
    y = keras.layers.Dropout(0.2)(y)

    # Fusion
    z = keras.layers.Concatenate()([x, y])
    z = keras.layers.Dense(256, activation='relu')(z)
    z = keras.layers.Dropout(0.3)(z)
    z = keras.layers.Dense(128, activation='relu')(z)
    z = keras.layers.Dropout(0.2)(z)
    z = keras.layers.Dense(64,  activation='relu')(z)
    output = keras.layers.Dense(3, activation='softmax', name='output')(z)

    return keras.Model(inputs=[seq_input, num_input], outputs=output)

# Preview parameter count
build_model().summary()

In [ ]:
# Cell 3.4 — Load dataset
print(f'[LOAD] Reading {DATA_PATH} ...')
df = pd.read_csv(DATA_PATH, low_memory=False)
df = df[df['label'].isin([0, 2])].copy()
print(f'       {len(df):,} rows  (SAFE + MALICIOUS only)')

X_num = df[FEATURE_COLUMNS].values.astype(np.float32)

print('[ENCODE] Tokenizing URLs (~1-2 min) ...')
t0 = time.time()
X_seq = np.array([encode_url(str(u)) for u in df['url']], dtype=np.int32)
y     = df['label'].values.astype(np.int32)
print(f'         Done in {time.time()-t0:.0f}s')
print(f'X_seq: {X_seq.shape}   X_num: {X_num.shape}   y: {y.shape}')

In [ ]:
# Cell 3.5 — GroupShuffleSplit 80/10/10 (no domain leakage)
def get_host(u):
    try:
        h = urlparse(u if '://' in u else 'http://' + u).netloc.split(':')[0].lower()
        return re.sub(r'^www\.', '', h)
    except Exception:
        return str(u)

print('Extracting domain groups ...')
groups = np.array([get_host(str(u)) for u in df['url']])

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, tmp_idx = next(gss1.split(X_seq, y, groups))

X_seq_tr,  X_seq_tmp  = X_seq[tr_idx],  X_seq[tmp_idx]
X_num_tr,  X_num_tmp  = X_num[tr_idx],  X_num[tmp_idx]
y_tr,      y_tmp      = y[tr_idx],      y[tmp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_idx, te_idx = next(gss2.split(X_seq_tmp, y_tmp, groups[tmp_idx]))

X_seq_val, X_seq_te = X_seq_tmp[val_idx], X_seq_tmp[te_idx]
X_num_val, X_num_te = X_num_tmp[val_idx], X_num_tmp[te_idx]
y_val,     y_te     = y_tmp[val_idx],     y_tmp[te_idx]

print(f'Train: {len(y_tr):,}   Val: {len(y_val):,}   Test: {len(y_te):,}')

In [ ]:
# Cell 3.6 — StandardScaler (fit on train only) + save
scaler    = StandardScaler()
X_num_tr  = scaler.fit_transform(X_num_tr).astype(np.float32)
X_num_val = scaler.transform(X_num_val).astype(np.float32)
X_num_te  = scaler.transform(X_num_te).astype(np.float32)

with open(SCALER_PATH, 'w') as f:
    json.dump({
        'n_features':    N_FEATURES,
        'feature_names': FEATURE_COLUMNS,
        'mean':          scaler.mean_.tolist(),
        'scale':         scaler.scale_.tolist(),
    }, f, indent=2)
print(f'✅ Scaler saved -> {SCALER_PATH}  (n_features={N_FEATURES})')

In [ ]:
# Cell 3.7 — Class weights, compile, train
raw_w = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
class_weight = dict(zip(np.unique(y_tr).tolist(), raw_w.tolist()))
if 1 not in class_weight:
    class_weight[1] = 1.0
print(f'Class weights: {class_weight}')

model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, monitor='val_loss', min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'best_model.keras'),
        save_best_only=True, monitor='val_accuracy',
    ),
]

print(f'\nTraining — GPU: {bool(tf.config.list_physical_devices("GPU"))}  batch={BATCH_SIZE}  max_epochs={EPOCHS} ...')
t0 = time.time()
history = model.fit(
    x={'seq_input': X_seq_tr, 'num_input': X_num_tr},
    y=y_tr,
    validation_data=({'seq_input': X_seq_val, 'num_input': X_num_val}, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)
print(f'\n⏱  Training done in {(time.time()-t0)/60:.1f} min  ({len(history.epoch)} epochs)')

In [ ]:
# Cell 3.8 — Training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history.history['loss'],     label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
ax2.plot(history.history['accuracy'],     label='train')
ax2.plot(history.history['val_accuracy'], label='val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig('models/cnn_training_curve.png', dpi=150)
plt.show()
print('Saved -> models/cnn_training_curve.png')

In [ ]:
# Cell 3.9 — Test set evaluation
loss, acc = model.evaluate(
    x={'seq_input': X_seq_te, 'num_input': X_num_te}, y=y_te, verbose=0
)
print(f'Test accuracy : {acc:.4f}  ({acc*100:.2f}%)   loss: {loss:.4f}')

y_pred = np.argmax(model.predict(
    {'seq_input': X_seq_te, 'num_input': X_num_te}, verbose=0
), axis=1)

print()
print(classification_report(y_te, y_pred, target_names=['SAFE','WARNING','MALICIOUS'], labels=[0,1,2]))
print('Confusion matrix  [rows=actual  cols=predicted]  labels=[SAFE, MALICIOUS]')
print(confusion_matrix(y_te, y_pred, labels=[0, 2]))

In [ ]:
# Cell 3.10 — Export to TFLite (dynamic range quantization)
# INT8 is NOT used here — incompatible with int32 Embedding layer on Windows TFLite.
print('[TFLITE] Converting CNN model (dynamic range quantization) ...')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_bytes)

kb = os.path.getsize(TFLITE_PATH) / 1024
status = '✅ Within 300 KB target' if kb <= 300 else f'⚠️  Exceeds 300 KB ({kb:.1f} KB)'
print(f'Saved -> {TFLITE_PATH}  ({kb:.1f} KB)  {status}')

In [ ]:
# Cell 3.11 — Validate TFLite inference
interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp_d = interp.get_input_details()
out_d = interp.get_output_details()

print('TFLite inputs:')
for d in inp_d:
    print(f'  {d["name"]:20s}  shape={d["shape"].tolist():15}  dtype={d["dtype"].__name__}')

for d in inp_d:
    if 'seq' in d['name']:
        interp.set_tensor(d['index'], X_seq_te[0:1].astype(np.int32))
    else:
        interp.set_tensor(d['index'], X_num_te[0:1].astype(np.float32))
interp.invoke()
out = interp.get_tensor(out_d[0]['index'])[0]
label_names = ['SAFE', 'WARNING', 'MALICIOUS']
print(f'\nOutput [p_safe, p_warning, p_malicious]: {out}')
print(f'Predicted : {label_names[np.argmax(out)]}')
print(f'True      : {label_names[y_te[0] if y_te[0] != 2 else 2]}')
print('\n✅ TFLite CNN validated')

---
## Section 4 — Autoencoder Training (Zero-day Anomaly Detector)

Trained on **benign URLs only**. A high MSE reconstruction error at inference = anomaly = possible zero-day.

Exports `models/autoencoder.tflite` and `models/anomaly_threshold.json`.

In [ ]:
# Cell 4.1 — Autoencoder constants and scaler
import os, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from feature_extractor import FEATURE_COLUMNS, N_FEATURES

AE_TFLITE    = 'models/autoencoder.tflite'
THRESHOLD_F  = 'models/anomaly_threshold.json'
LATENT_DIM   = 10
AE_BATCH     = 256
AE_EPOCHS    = 30
SHORT_MAX    = 30
LONG_MIN     = 80
SEED         = 42

# Load scaler params from Section 3 (must have run Section 3 first)
with open(SCALER_PATH) as f:
    sp = json.load(f)
assert sp['n_features'] == N_FEATURES
ae_mean  = np.array(sp['mean'],  dtype=np.float32)
ae_scale = np.array(sp['scale'], dtype=np.float32)
print(f'✅ Scaler loaded  (n_features={N_FEATURES})')

In [ ]:
# Cell 4.2 — Load benign URLs and scale
df_all    = pd.read_csv(DATA_PATH, low_memory=False)
benign_df = df_all[df_all['label'] == 0].copy()
print(f'Benign rows : {len(benign_df):,}')

X_ae     = benign_df[FEATURE_COLUMNS].values.astype(np.float32)
url_lens = benign_df['url'].str.len().values
X_ae_sc  = ((X_ae - ae_mean) / ae_scale).astype(np.float32)

X_tr_ae, X_val_ae, ul_tr, ul_val = train_test_split(
    X_ae_sc, url_lens, test_size=0.1, random_state=SEED
)
print(f'AE train: {len(X_tr_ae):,}   val: {len(X_val_ae):,}')

In [ ]:
# Cell 4.3 — Build and train autoencoder
def build_autoencoder(d=N_FEATURES):
    inp = keras.Input(shape=(d,), name='ae_input')
    x   = keras.layers.Dense(16, activation='relu')(inp)
    x   = keras.layers.Dense(LATENT_DIM, activation='relu', name='latent')(x)
    x   = keras.layers.Dense(16, activation='relu')(x)
    out = keras.layers.Dense(d,  activation='linear', name='ae_output')(x)
    return keras.Model(inputs=inp, outputs=out, name='autoencoder')

np.random.seed(SEED); tf.random.set_seed(SEED)
ae = build_autoencoder()
ae.summary()
ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

ae_cb = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, monitor='val_loss'),
]

print('\nTraining autoencoder ...')
t0 = time.time()
ae_hist = ae.fit(
    X_tr_ae, X_tr_ae,
    validation_data=(X_val_ae, X_val_ae),
    epochs=AE_EPOCHS, batch_size=AE_BATCH,
    callbacks=ae_cb, verbose=1,
)
print(f'⏱  AE done in {(time.time()-t0)/60:.1f} min  ({len(ae_hist.epoch)} epochs)')

In [ ]:
# Cell 4.4 — Adaptive 95th-percentile thresholds per URL length category
def url_cat(l):
    return 'short' if l < SHORT_MAX else ('medium' if l <= LONG_MIN else 'long')

val_recon  = ae.predict(X_val_ae, verbose=0)
mse_scores = np.mean((X_val_ae - val_recon) ** 2, axis=1)

thresholds = {}
for cat in ['short', 'medium', 'long']:
    mask = np.array([url_cat(l) == cat for l in ul_val])
    thresholds[cat] = float(np.percentile(
        mse_scores[mask] if mask.sum() > 0 else mse_scores, 95
    ))
    print(f'  {cat:6s} : {int(mask.sum()):6,} samples   threshold = {thresholds[cat]:.6f}')

with open(THRESHOLD_F, 'w') as f:
    json.dump({
        'thresholds': thresholds,
        'short_max': SHORT_MAX,
        'long_min':  LONG_MIN,
        'description': 'Adaptive 95th-percentile MSE thresholds per URL length category',
    }, f, indent=2)
print(f'\n✅ Thresholds saved -> {THRESHOLD_F}')

In [ ]:
# Cell 4.5 — Export autoencoder to TFLite INT8
print('[TFLITE] Converting autoencoder to TFLite INT8 ...')

def ae_rep_dataset():
    idx = np.random.choice(len(X_tr_ae), 200, replace=False)
    def gen():
        for i in idx:
            yield [X_tr_ae[i:i+1].astype(np.float32)]
    return gen

ae_conv = tf.lite.TFLiteConverter.from_keras_model(ae)
ae_conv.optimizations = [tf.lite.Optimize.DEFAULT]
ae_conv.representative_dataset = ae_rep_dataset()
ae_conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
ae_conv.inference_input_type  = tf.float32
ae_conv.inference_output_type = tf.float32

ae_bytes = ae_conv.convert()
with open(AE_TFLITE, 'wb') as f:
    f.write(ae_bytes)

kb = os.path.getsize(AE_TFLITE) / 1024
print(f'✅ Saved -> {AE_TFLITE}  ({kb:.1f} KB)')

In [ ]:
# Cell 4.6 — Validate autoencoder TFLite
ae_interp = tf.lite.Interpreter(model_path=AE_TFLITE)
ae_interp.allocate_tensors()
ai = ae_interp.get_input_details()[0]
ao = ae_interp.get_output_details()[0]
print(f'AE input  : shape={ai["shape"].tolist()}  dtype={ai["dtype"].__name__}')
print(f'AE output : shape={ao["shape"].tolist()}  dtype={ao["dtype"].__name__}')

sample = X_val_ae[0:1].astype(np.float32)
ae_interp.set_tensor(ai['index'], sample)
ae_interp.invoke()
recon = ae_interp.get_tensor(ao['index'])
mse   = float(np.mean((sample - recon) ** 2))
cat   = url_cat(ul_val[0])
print(f'\nSample MSE  : {mse:.6f}')
print(f'Category    : {cat}')
print(f'Threshold   : {thresholds[cat]:.6f}')
print(f'Is anomaly  : {mse > thresholds[cat]}')
print('\n✅ Autoencoder TFLite validated')

---
## Section 5 — Copy Assets to Android App

In [ ]:
# Cell 5.1 — Copy trained models into Android assets
import shutil, os

ANDROID_ASSETS = '../SafeLink Android/app/src/main/assets'

FILES = [
    ('models/safelink_model.tflite',  'safelink_model.tflite'),
    ('models/autoencoder.tflite',     'autoencoder.tflite'),
    ('models/scaler_params.json',     'scaler_params.json'),
    ('models/anomaly_threshold.json', 'anomaly_threshold.json'),
]

if not os.path.isdir(ANDROID_ASSETS):
    print(f'⚠️  Android assets folder not found at:')
    print(f'   {os.path.abspath(ANDROID_ASSETS)}')
    print('   Copy the 4 files from ml_pipeline/models/ manually into the assets folder.')
else:
    print(f'Copying to: {os.path.abspath(ANDROID_ASSETS)}\n')
    for src, dst in FILES:
        if os.path.exists(src):
            dest = os.path.join(ANDROID_ASSETS, dst)
            shutil.copy2(src, dest)
            kb = os.path.getsize(dest) / 1024
            print(f'  ✅  {dst:35s}  ({kb:.1f} KB)')
        else:
            print(f'  ❌  {src}  — not found, train first')
    print('\n✅ Done')

---
## Section 6 — Final Summary

In [ ]:
# Cell 6.1 — Output summary
print('=' * 58)
print('  SafeLink PRD v5.1 — Training Complete')
print('=' * 58)

for path, label in [
    ('models/safelink_model.tflite',   'CNN model      '),
    ('models/autoencoder.tflite',      'Autoencoder    '),
    ('models/scaler_params.json',      'Scaler params  '),
    ('models/anomaly_threshold.json',  'AE thresholds  '),
    ('models/best_model.keras',        'Keras checkpoint'),
    ('models/cnn_training_curve.png',  'Training curve '),
]:
    exists = os.path.exists(path)
    kb = f'  ({os.path.getsize(path)/1024:.1f} KB)' if exists else ''
    print(f'  {"✅" if exists else "❌"}  {label}  {path}{kb}')

print()
print('Next steps:')
print('  1. Check test accuracy above (target: > 90%)')
print('  2. Open Android Studio → Build → Run on device')
print('  3. Test with ManualTestActivity for real-world URL checks')